# Python declarativo vs. imperativo

**Imperativo** = describir *cómo* se hace algo, paso a paso: crear un
contenedor vacío, recorrer, ir agregando, manejar índices y banderas.

**Declarativo** = describir *qué* se quiere obtener y dejar que Python
(o la librería estándar) resuelva el cómo.

Cada ejercicio de este notebook muestra **las dos versiones en la misma celda**:
primero la imperativa, después la declarativa, y al final una comparación que
comprueba que producen exactamente el mismo resultado.

Cada celda de código empieza con un **bloque de referencia en comentarios**
(`MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA`) que documenta cada método,
función o sintaxis que aparece: firma, qué devuelve y la trampa que conviene
conocer. Leer primero el bloque, después el código, y volver al bloque cuando
algo no cierre.

Los ejercicios van de menor a mayor complejidad. Requiere **Python 3.10+**
(por `match/case` y `zip(strict=True)`).

In [1]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# from modulo import nombre
#     Trae un nombre al espacio de nombres actual. Cada uno se explica en la
#     celda donde se usa. Regla senior: importar nombres concretos, nunca
#     `from modulo import *` (oculta de dónde viene cada cosa y pisa nombres).
# sys.version
#     str con la versión del intérprete y datos de compilación.
#     .split()[0] se queda con "3.14.3". Para comparar versiones usar la tupla
#     sys.version_info >= (3, 10), nunca comparar el string.
# print(*objetos, sep=" ", end="\n")
#     Convierte cada objeto con str() y los une con `sep`. Devuelve None.
# ============================================================================

# Imports usados a lo largo del notebook. Ejecutar esta celda primero.
from collections import Counter, defaultdict
from contextlib import contextmanager
from dataclasses import dataclass, field
from enum import Enum
from functools import lru_cache, reduce
from itertools import accumulate, chain, groupby, islice
from operator import attrgetter, itemgetter
import sys
import time

print("Python", sys.version.split()[0])

Python 3.14.3


## 1. List comprehension

La puerta de entrada al estilo declarativo. La versión imperativa necesita
tres cosas que pueden salir mal: una lista vacía inicial, un bucle y un
`append`. La declarativa es **una sola expresión**: no hay estado intermedio
que quede a medias si algo falla.

In [2]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# range(inicio, fin, paso=1)
#     Secuencia PEREZOSA de enteros; `fin` queda EXCLUIDO: range(1, 6) -> 1..5.
#     No construye una lista: range(10**9) ocupa lo mismo que range(3).
#     Soporta len(), indexación, slicing y `in` en O(1).
# list.append(x)
#     Agrega x al final, IN PLACE. Devuelve None: `lista = lista.append(x)`
#     deja `lista` en None. Es el error más común de principiante.
# x ** y
#     Potencia. Con enteros da int exacto (sin límite de tamaño); con float
#     puede perder precisión. 2 ** -1 -> 0.5 (int elevado a negativo da float).
# [expresion for variable in iterable]
#     List comprehension: evalúa `expresion` por cada elemento y arma la lista
#     completa en memoria. La variable del for NO se filtra hacia fuera
#     (tiene su propio scope desde Python 3), a diferencia del bucle for.
# ==
#     Igualdad por VALOR. Dos listas distintas con los mismos elementos son ==.
#     `is` compara identidad (mismo objeto); usar `is` solo con None/True/False.
# ============================================================================

# --- Imperativo: cómo ---
cuadrados_imp = []              # 1) contenedor vacío
for n in range(1, 6):           # 2) recorrer
    cuadrados_imp.append(n ** 2)  # 3) ir agregando

# --- Declarativo: qué ---
# "los cuadrados de los números del 1 al 5"
cuadrados_dec = [n ** 2 for n in range(1, 6)]

print("imperativo :", cuadrados_imp)
print("declarativo:", cuadrados_dec)
print("iguales    :", cuadrados_imp == cuadrados_dec)

imperativo : [1, 4, 9, 16, 25]
declarativo: [1, 4, 9, 16, 25]
iguales    : True


## 2. Comprehension con filtro

El `if` va **después** del `for`: decide qué elementos entran al resultado.
Sustituye al `if ...: lista.append(...)` dentro del bucle.

In [3]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# a % b
#     Módulo: resto de la división entera. Con el signo del DIVISOR:
#     -7 % 2 -> 1 (no -1 como en C/Java). n % 2 == 0 es "es par" para
#     cualquier entero, positivo o negativo.
# [expresion for variable in iterable if condicion]
#     Comprehension con FILTRO. Solo los elementos que cumplen `condicion`
#     llegan a `expresion`. Se pueden encadenar varios `if` seguidos (se
#     combinan con AND implícito), aunque `and` explícito se lee mejor.
# ============================================================================

numeros = [4, 7, 10, 13, 16, 21, 24]

# --- Imperativo ---
pares_imp = []
for n in numeros:
    if n % 2 == 0:
        pares_imp.append(n)

# --- Declarativo ---
# "n, para cada n de numeros, si n es par"
pares_dec = [n for n in numeros if n % 2 == 0]

print("imperativo :", pares_imp)
print("declarativo:", pares_dec)
print("iguales    :", pares_imp == pares_dec)

imperativo : [4, 10, 16, 24]
declarativo: [4, 10, 16, 24]
iguales    : True


## 3. Dict comprehension

Misma sintaxis, pero con `clave: valor`. Sirve para transformar, indexar o
invertir diccionarios. Lo imperativo obliga a repetir el nombre del dict de
salida en cada asignación; un typo ahí es un bug silencioso.

In [4]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# dict.items()
#     Vista PEREZOSA de pares (clave, valor). Refleja cambios posteriores en el
#     dict y no copia nada. Modificar el dict mientras se itera lanza
#     RuntimeError: iterar sobre list(d.items()) si hay que mutar.
# int(x)
#     Convierte a entero TRUNCANDO hacia cero: int(1080.9) -> 1080,
#     int(-0.9) -> 0. No redondea. Para redondear usar round().
# {clave: valor for ... in ...}
#     Dict comprehension. Si dos iteraciones producen la misma clave, GANA LA
#     ÚLTIMA en silencio: al invertir un dict con valores repetidos se pierden
#     entradas sin aviso.
# ============================================================================

precios = {"pan": 1200, "leche": 990, "queso": 4500}

# --- Imperativo ---
con_descuento_imp = {}
for producto, precio in precios.items():
    con_descuento_imp[producto] = int(precio * 0.9)

invertido_imp = {}
for producto, precio in precios.items():
    invertido_imp[precio] = producto

# --- Declarativo ---
con_descuento_dec = {prod: int(p * 0.9) for prod, p in precios.items()}
invertido_dec = {p: prod for prod, p in precios.items()}

print("descuento imperativo :", con_descuento_imp)
print("descuento declarativo:", con_descuento_dec)
print("invertido declarativo:", invertido_dec)
print("iguales:", con_descuento_imp == con_descuento_dec, invertido_imp == invertido_dec)

descuento imperativo : {'pan': 1080, 'leche': 891, 'queso': 4050}
descuento declarativo: {'pan': 1080, 'leche': 891, 'queso': 4050}
invertido declarativo: {1200: 'pan', 990: 'leche', 4500: 'queso'}
iguales: True True


## 4. Set comprehension

El `set` elimina duplicados **por definición**. En la versión imperativa hay
que preguntar `if x not in resultado` en cada vuelta: además de ruido, sobre
una lista esa comprobación es O(n), así que el bucle completo es O(n²).

In [5]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# str.split(sep=None, maxsplit=-1)
#     Sin argumentos divide por CUALQUIER espacio en blanco (espacios, tabs,
#     saltos) y descarta los vacíos: "  a  b ".split() -> ['a', 'b'].
#     Con sep explícito NO descarta: "a,,b".split(",") -> ['a', '', 'b'].
# cadena[0]
#     Indexación: primer carácter. Con cadena vacía lanza IndexError;
#     cadena[0:1] devuelve "" sin error. Un str es inmutable: no admite
#     asignación por índice.
# x not in coleccion
#     Pertenencia negada. En list/tuple es O(n) (recorre); en set/dict es O(1)
#     (hash). Por eso "deduplicar con lista" es cuadrático.
# {expresion for ... in ...}
#     Set comprehension. Elementos únicos, SIN ORDEN garantizado, y solo
#     hashables (no se pueden meter listas ni dicts).
# sorted(iterable, /, *, key=None, reverse=False)
#     Devuelve una LISTA NUEVA ordenada de cualquier iterable (set, dict,
#     generador...). list.sort() en cambio ordena in place y devuelve None.
# ============================================================================

frase = "el declarativo describe el resultado deseado"

# --- Imperativo ---
iniciales_imp = []
for palabra in frase.split():
    inicial = palabra[0]
    if inicial not in iniciales_imp:   # comprobación manual de duplicados
        iniciales_imp.append(inicial)

# --- Declarativo ---
iniciales_dec = {palabra[0] for palabra in frase.split()}

print("imperativo :", sorted(iniciales_imp))
print("declarativo:", sorted(iniciales_dec))
print("iguales    :", sorted(iniciales_imp) == sorted(iniciales_dec))

imperativo : ['d', 'e', 'r']
declarativo: ['d', 'e', 'r']
iguales    : True


## 5. Generator expression (evaluación perezosa)

Con **paréntesis** en vez de corchetes no se construye la lista completa: se
produce un elemento a la vez, bajo demanda. La diferencia no es de estilo sino
de memoria — mirá los bytes que ocupa cada uno.

El equivalente imperativo de un generador no es un bucle, es escribir una
clase iteradora completa con `__iter__` y `__next__`.

In [6]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# Protocolo de iteración
#     __iter__(self) -> devuelve el iterador (aquí `self`).
#     __next__(self) -> devuelve el siguiente valor o lanza StopIteration.
#     for, list(), sum(), etc. llaman a iter() y luego next() hasta la excepción.
#     Un iterador se AGOTA: la segunda vuelta sobre él no produce nada.
# next(iterador, default)
#     Pide el siguiente valor. Sin `default` lanza StopIteration al agotarse;
#     con `default` lo devuelve. Los generadores tienen __next__, las listas
#     no: next([1, 2]) -> TypeError (hay que hacer next(iter(lista))).
# (expresion for variable in iterable)
#     Generator expression: NO calcula nada al crearse; produce un valor por
#     cada next(). Es de un solo uso. Como argumento único de una función los
#     paréntesis se omiten: sum(n for n in ...).
# sys.getsizeof(objeto)
#     Bytes del objeto EN SÍ, sin contar lo que referencia. Una lista de un
#     millón de ints reporta ~8 MB (punteros); los ints van aparte. Sirve para
#     comparar contenedores, no para medir memoria total.
# sum(iterable, /, start=0)
#     Suma acumulando desde `start`. No usar para concatenar strings (O(n²));
#     para eso "".join(). Con floats usar math.fsum si importa la precisión.
# raise StopIteration
#     Señal de "no hay más". Dentro de un generador (def con yield) NO se lanza
#     a mano: se hace `return`; lanzarla se convierte en RuntimeError (PEP 479).
# ============================================================================

# --- Imperativo: la clase iteradora que el generador nos ahorra ---
class CuadradosImp:
    def __init__(self, tope):
        self.tope = tope
        self.actual = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.actual >= self.tope:
            raise StopIteration
        valor = self.actual ** 2
        self.actual += 1
        return valor


# --- Declarativo ---
cuadrados_gen = (n ** 2 for n in range(1_000_000))

it_imp = CuadradosImp(1_000_000)
print("imperativo :", next(it_imp), next(it_imp))
print("declarativo:", next(cuadrados_gen), next(cuadrados_gen))

# El costo en memoria: lista completa vs. generador
lista = [n ** 2 for n in range(1_000_000)]
print("lista     :", sys.getsizeof(lista), "bytes")
print("generador :", sys.getsizeof(n ** 2 for n in range(1_000_000)), "bytes")

# Consumido por una función agregadora, nunca materializa la lista
print("suma perezosa:", sum(n ** 2 for n in range(1, 101)))

imperativo : 0 1
declarativo: 0 1
lista     : 8448728 bytes
generador : 208 bytes
suma perezosa: 338350


## 6. Funciones agregadoras: `sum`, `min`, `max`, `any`, `all`

Reemplazan bucles con acumuladores y banderas booleanas. Fijate en el bug
clásico que evita `min`/`max`: inicializar el acumulador en `0` en vez de en
el primer elemento (con notas negativas o todas mayores que 0, el resultado
sale mal).

`any` y `all` además **cortocircuitan**: dejan de evaluar apenas saben la
respuesta.

In [7]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# sum(iterable, start=0)      -> ver celda 5.
# len(objeto)
#     Cantidad de elementos, O(1) en list/dict/set/str (está cacheada).
#     Un generador NO tiene len: hay que consumirlo o contar de otra forma.
# min(iterable, *, key=None, default=...) / max(...)
#     Menor/mayor elemento. Con iterable VACÍO lanza ValueError salvo que se
#     pase `default`. `key` permite comparar por un criterio derivado:
#     max(alumnos, key=lambda a: a["nota"]) devuelve el alumno, no la nota.
# any(iterable)
#     True si ALGÚN elemento es verdadero. any([]) -> False. Cortocircuita:
#     para en el primer True. Combinado con un generador no evalúa lo que
#     sobra: any(caro(x) for x in datos) puede llamar a caro() una sola vez.
# all(iterable)
#     True si TODOS son verdaderos. all([]) -> True (verdad vacía: cuidado al
#     validar listas que pueden llegar vacías). Cortocircuita en el primer False.
# break
#     Sale del bucle más interno. En la versión imperativa hace falta para
#     imitar el cortocircuito; sin él se recorre todo aunque ya se sepa.
# +=
#     Asignación aumentada. En ints/floats crea un objeto nuevo; en listas
#     (`lista += [x]`) muta IN PLACE la lista original (efecto visible desde
#     otras referencias). No es lo mismo que `lista = lista + [x]`.
# ============================================================================

notas = [6.5, 4.0, 7.0, 3.2, 5.8]

# --- Imperativo ---
total = 0
for n in notas:
    total += n
promedio_imp = total / len(notas)

minimo_imp = notas[0]           # inicializar en 0 acá sería el bug clásico
maximo_imp = notas[0]
for n in notas:
    if n < minimo_imp:
        minimo_imp = n
    if n > maximo_imp:
        maximo_imp = n

hay_reprobada_imp = False       # bandera
for n in notas:
    if n < 4.0:
        hay_reprobada_imp = True
        break

todas_aprobadas_imp = True      # otra bandera, con la lógica invertida
for n in notas:
    if n < 4.0:
        todas_aprobadas_imp = False
        break

# --- Declarativo ---
promedio_dec = sum(notas) / len(notas)
minimo_dec, maximo_dec = min(notas), max(notas)
hay_reprobada_dec = any(n < 4.0 for n in notas)
todas_aprobadas_dec = all(n >= 4.0 for n in notas)

print("promedio :", promedio_imp, promedio_dec)
print("min/max  :", (minimo_imp, maximo_imp), (minimo_dec, maximo_dec))
print("any      :", hay_reprobada_imp, hay_reprobada_dec)
print("all      :", todas_aprobadas_imp, todas_aprobadas_dec)

promedio : 5.3 5.3
min/max  : (3.2, 7.0) (3.2, 7.0)
any      : True True
all      : False False


## 7. `sorted(key=...)`: declarar el criterio, no el algoritmo

La versión imperativa exige **escribir un algoritmo de ordenamiento**. Aquí va
un selection sort, que además es O(n²) e inestable. `sorted` usa Timsort
(O(n log n), estable) y solo hay que decirle *por qué valor comparar*.

In [8]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# list.copy()
#     Copia SUPERFICIAL: lista nueva, mismos objetos dentro. Equivale a
#     lista[:]. Para copiar también los dicts internos: copy.deepcopy().
# a, b = b, a
#     Intercambio sin variable temporal. Primero se evalúa la tupla derecha
#     completa, después se asigna: por eso funciona.
# sorted(iterable, /, *, key=None, reverse=False)
#     Timsort: O(n log n) y ESTABLE (a igualdad de key conserva el orden
#     original). `key` recibe UNA función de un argumento que se llama UNA
#     vez por elemento (no un comparador de dos argumentos como en Java/JS;
#     para eso existe functools.cmp_to_key).
#     reverse=True mantiene la estabilidad (no es lo mismo que invertir después).
# lambda argumentos: expresion
#     Función anónima de UNA expresión (sin statements, sin return). Ideal
#     como `key`. Si supera una línea, mejor `def` con nombre.
# key=lambda a: (-a["nota"], a["edad"])
#     Tupla como criterio: compara elemento a elemento (lexicográfico). El
#     signo menos invierte SOLO ese campo; con strings no se puede negar:
#     ahí se ordena dos veces (sorted es estable) o se usa reverse por pasos.
# operator.itemgetter(*claves)
#     Fabrica una función obj -> obj[clave]. Con varias claves devuelve una
#     tupla: itemgetter("nota", "edad"). Más rápido que la lambda (C puro).
# operator.attrgetter(*nombres)
#     Igual pero para atributos: obj -> obj.nombre. Admite rutas con punto:
#     attrgetter("direccion.ciudad").
# ============================================================================

alumnos = [
    {"nombre": "Ana", "nota": 6.8, "edad": 21},
    {"nombre": "Beto", "nota": 4.2, "edad": 25},
    {"nombre": "Carla", "nota": 6.8, "edad": 19},
]

# --- Imperativo: selection sort por nota descendente ---
por_nota_imp = alumnos.copy()
for i in range(len(por_nota_imp)):
    mayor = i
    for j in range(i + 1, len(por_nota_imp)):
        if por_nota_imp[j]["nota"] > por_nota_imp[mayor]["nota"]:
            mayor = j
    por_nota_imp[i], por_nota_imp[mayor] = por_nota_imp[mayor], por_nota_imp[i]

# --- Declarativo ---
por_nota_dec = sorted(alumnos, key=lambda a: a["nota"], reverse=True)

print("imperativo :", [a["nombre"] for a in por_nota_imp])
print("declarativo:", [a["nombre"] for a in por_nota_dec])

# Varios criterios: la key devuelve una tupla.
# Nota descendente (de ahí el signo menos) y, a igual nota, edad ascendente.
ranking = sorted(alumnos, key=lambda a: (-a["nota"], a["edad"]))
print("ranking    :", [a["nombre"] for a in ranking])

# itemgetter/attrgetter: la versión sin lambda, más corta y más rápida
print("por edad   :", [a["nombre"] for a in sorted(alumnos, key=itemgetter("edad"))])

imperativo : ['Ana', 'Carla', 'Beto']
declarativo: ['Ana', 'Carla', 'Beto']
ranking    : ['Carla', 'Ana', 'Beto']
por edad   : ['Carla', 'Ana', 'Beto']


## 8. `map()` y `filter()`

Son el equivalente funcional de las comprehensions. En Python moderno suele
preferirse la comprehension por legibilidad, **salvo** cuando ya existe una
función con nombre que aplicar: ahí `map(str.upper, ...)` gana a
`[s.upper() for s in ...]`.

In [9]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# map(funcion, iterable, *mas_iterables)
#     Aplica `funcion` a cada elemento y devuelve un ITERADOR perezoso (no
#     una lista: hay que hacer list() o iterar). Con varios iterables pasa un
#     elemento de cada uno: map(pow, [2, 3], [10, 2]) -> 1024, 9.
# filter(funcion_o_None, iterable)
#     Devuelve un iterador con los elementos donde funcion(x) es verdadero.
#     filter(None, iterable) elimina los falsy (0, "", None, [] ...).
#     Para "los que NO cumplen" existe itertools.filterfalse.
# list(iterable)
#     Materializa cualquier iterable en una lista nueva. Consume generadores
#     e iteradores de map/filter: después de list() quedan vacíos.
# str.upper
#     Método SIN paréntesis: se pasa la función misma, no su resultado.
#     str.upper("ana") == "ana".upper(). Cualquier método puede usarse así
#     como función de primera clase: map(str.strip, lineas).
# c * 9 / 5 + 32
#     Precedencia: * y / se evalúan de izquierda a derecha antes que +.
#     `/` siempre devuelve float en Python 3 (10 / 2 -> 5.0); `//` es
#     división entera.
# ============================================================================

temperaturas_c = [0, 18, 25, 37, 100]

# --- Imperativo ---
fahrenheit_imp = []
for c in temperaturas_c:
    fahrenheit_imp.append(c * 9 / 5 + 32)

calurosas_imp = []
for c in temperaturas_c:
    if c > 20:
        calurosas_imp.append(c)

# --- Declarativo ---
fahrenheit_dec = list(map(lambda c: c * 9 / 5 + 32, temperaturas_c))
calurosas_dec = list(filter(lambda c: c > 20, temperaturas_c))

print("map    :", fahrenheit_imp, fahrenheit_dec)
print("filter :", calurosas_imp, calurosas_dec)

# map con una función ya existente: sin lambda de por medio
print("sin lambda:", list(map(str.upper, ["ana", "beto", "carla"])))

map    : [32.0, 64.4, 77.0, 98.6, 212.0] [32.0, 64.4, 77.0, 98.6, 212.0]
filter : [25, 37, 100] [25, 37, 100]
sin lambda: ['ANA', 'BETO', 'CARLA']


## 9. `zip()` y `enumerate()`

Recorrer con `for i in range(len(lista))` es frágil: `IndexError` si las listas
tienen distinto largo, y el índice es una variable más que mantener. `zip` y
`enumerate` eliminan el índice del código.

`zip` se detiene en el iterable más corto; con `strict=True` (3.10+) **falla
temprano** en vez de truncar en silencio.

In [10]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# zip(*iterables, strict=False)
#     Empareja elemento a elemento y devuelve un iterador de TUPLAS. Se
#     detiene en el iterable MÁS CORTO en silencio. strict=True (3.10+) lanza
#     ValueError si los largos difieren: preferirlo cuando el desajuste es un
#     bug y no algo esperado. zip(*pares) "des-zipea" (transpone).
# dict(iterable_de_pares)
#     Construye un dict desde pares (clave, valor): dict(zip(claves, valores))
#     es el idioma para armar un dict de dos listas paralelas.
# enumerate(iterable, start=0)
#     Devuelve pares (índice, elemento). `start` fija el primer índice: para
#     mostrar "1º, 2º, 3º" usar start=1 en vez de sumar 1 a mano.
# range(len(lista))
#     El ANTIPATRÓN que zip/enumerate reemplazan: obliga a indexar y falla con
#     iterables sin len (generadores). Solo se justifica si hace falta el
#     índice para escribir en otra lista.
# f"texto {expresion}"
#     f-string: evalúa la expresión y la interpola. Admite formato después de
#     dos puntos: f"{x:.2f}", f"{n:>5}", f"{fecha:%Y-%m-%d}", y f"{x=}"
#     imprime "x=valor" para depurar.
# try / except ValueError as error
#     Captura SOLO esa excepción (y subclases); las demás siguen subiendo.
#     `as error` da acceso al mensaje. Capturar `Exception` a secas oculta
#     bugs; capturar `BaseException` o usar `except:` desnudo es peor todavía.
# ============================================================================

nombres = ["Ana", "Beto", "Carla"]
puntajes = [95, 72, 88]

# --- Imperativo ---
tabla_imp = {}
for i in range(len(nombres)):        # el índice como variable a mantener
    tabla_imp[nombres[i]] = puntajes[i]

posicion = 1                         # contador manual
for nombre in nombres:
    print(f"imperativo : {posicion}º lugar: {nombre}")
    posicion += 1

# --- Declarativo ---
tabla_dec = dict(zip(nombres, puntajes))
for posicion, nombre in enumerate(nombres, start=1):
    print(f"declarativo: {posicion}º lugar: {nombre}")

print("tabla iguales:", tabla_imp == tabla_dec, tabla_dec)

# strict=True detecta el desajuste de largos en vez de truncar
try:
    list(zip(nombres, [95, 72], strict=True))
except ValueError as error:
    print("strict=True avisa:", error)

imperativo : 1º lugar: Ana
imperativo : 2º lugar: Beto
imperativo : 3º lugar: Carla
declarativo: 1º lugar: Ana
declarativo: 2º lugar: Beto
declarativo: 3º lugar: Carla
tabla iguales: True {'Ana': 95, 'Beto': 72, 'Carla': 88}
strict=True avisa: zip() argument 2 is shorter than argument 1


## 10. Desempaquetado (*unpacking*)

Describir la **forma** del dato en vez de accederlo por posición. El `*`
captura "todo el resto", y `**` fusiona diccionarios sin `update()` ni
copias manuales.

In [11]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# a, b = iterable
#     Desempaquetado: exige EXACTAMENTE tantos elementos como variables, si no
#     lanza ValueError. Funciona con cualquier iterable, no solo tuplas.
# primero, *resto, ultimo = iterable
#     Desempaquetado extendido (PEP 3132): la variable con * captura el resto
#     como LISTA (aunque la fuente sea tupla). Solo puede haber un * por
#     asignación. Con exactamente 2 elementos, `resto` queda [].
# lista[-1]
#     Índice negativo: cuenta desde el final. lista[-1] es el último,
#     lista[-2] el penúltimo. Con lista vacía lanza IndexError.
# lista[1:-1]
#     Slicing [inicio:fin]: `fin` excluido, ambos opcionales, nunca lanza
#     IndexError (recorta silenciosamente). Devuelve una lista NUEVA
#     (copia superficial). [::-1] invierte, [::2] uno de cada dos.
# {**a, **b}
#     Fusión de dicts (PEP 448) en uno NUEVO. Con claves repetidas gana la
#     de más a la DERECHA. Desde 3.9 también existe `a | b` (mismo efecto) y
#     `a |= b` (in place). Es copia superficial: los valores no se clonan.
# [*a, *b]
#     Lo mismo para secuencias: concatena iterables de cualquier tipo en una
#     lista. [*range(3), *"ab"] mezcla un range con un str sin conversiones.
# dict.items()      -> ver celda 3.
# ============================================================================

punto = (3, 7)
lista = [10, 20, 30, 40, 50]

# --- Imperativo ---
x_imp = punto[0]
y_imp = punto[1]

primero_imp = lista[0]
ultimo_imp = lista[-1]
intermedios_imp = lista[1:-1]

base = {"host": "localhost", "puerto": 5432}
config_imp = {}
for clave, valor in base.items():        # copiar a mano
    config_imp[clave] = valor
config_imp["puerto"] = 5433              # sobrescribir
config_imp["debug"] = True

# --- Declarativo ---
x_dec, y_dec = punto
primero_dec, *intermedios_dec, ultimo_dec = lista
config_dec = {**base, "puerto": 5433, "debug": True}  # con clave repetida gana la última

print("punto  :", (x_imp, y_imp), (x_dec, y_dec))
print("bordes :", primero_dec, intermedios_dec, ultimo_dec)
print("config :", config_imp == config_dec, config_dec)
print("mezcla :", [*range(3), *"ab"])

punto  : (3, 7) (3, 7)
bordes : 10 [20, 30, 40] 50
config : True {'host': 'localhost', 'puerto': 5433, 'debug': True}
mezcla : [0, 1, 2, 'a', 'b']


## 11. Expresión condicional (ternario) en comprehensions

⚠️ El error clásico es la **posición del `if`**:

| Forma | Qué hace |
|---|---|
| `[x if cond else y for ...]` | `if` **antes** → *transforma*: elige qué valor sale |
| `[x for ... if cond]` | `if` **después** → *filtra*: elige qué elementos entran |

La primera devuelve siempre la misma cantidad de elementos; la segunda, menos.

In [12]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# valor_si_true if condicion else valor_si_false
#     Expresión condicional (ternario). Es una EXPRESIÓN: devuelve un valor y
#     puede ir dentro de otra expresión. Solo evalúa la rama elegida.
#     Orden de lectura: primero se mira `condicion`, aunque esté en el medio.
# [x if c else y for ...]   vs   [x for ... if c]
#     El primer `if` es parte de la EXPRESIÓN (ternario, obligatorio el else).
#     El segundo `if` es la cláusula de FILTRO de la comprehension (sin else;
#     poner `else` ahí es SyntaxError). Se pueden combinar: el filtro se
#     aplica primero, el ternario transforma lo que pasó.
# round(numero, ndigits=None)
#     Redondea. Sin ndigits devuelve int; con ndigits devuelve float.
#     Usa "redondeo bancario" (al par más cercano en empates):
#     round(0.5) -> 0, round(1.5) -> 2, round(2.5) -> 2. Y con floats hay
#     sorpresas de representación: round(2.675, 2) -> 2.67. Para dinero:
#     decimal.Decimal con ROUND_HALF_UP.
# ============================================================================

notas_finales = [6.8, 3.9, 4.0, 2.5, 7.0]

# --- Imperativo ---
estado_imp = []
for n in notas_finales:
    if n >= 4.0:
        estado_imp.append("aprobado")
    else:
        estado_imp.append("reprobado")

aprobados_imp = []
for n in notas_finales:
    if n >= 4.0:
        aprobados_imp.append(n)

# --- Declarativo ---
estado_dec = ["aprobado" if n >= 4.0 else "reprobado" for n in notas_finales]  # transforma
aprobados_dec = [n for n in notas_finales if n >= 4.0]                          # filtra

print("transforma (5 elementos):", estado_imp == estado_dec, estado_dec)
print("filtra     (3 elementos):", aprobados_imp == aprobados_dec, aprobados_dec)

# Ambos a la vez: el if final filtra, el ternario transforma lo que sobrevive
print("combinado:", [round(n) if n >= 4.0 else 0 for n in notas_finales if n > 3.0])

transforma (5 elementos): True ['aprobado', 'reprobado', 'aprobado', 'reprobado', 'aprobado']
filtra     (3 elementos): True [6.8, 4.0, 7.0]
combinado: [7, 0, 4, 7]


## 12. Comprehensions anidadas: aplanar y transponer

Los `for` de una comprehension se leen en el **mismo orden** que los bucles
anidados equivalentes: primero la fila, después el elemento dentro de esa fila.
Para transponer, `zip(*matriz)` es la forma idiomática.

In [13]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# [x for fila in matriz for x in fila]
#     Comprehension con DOS for: se leen en el mismo orden que si estuvieran
#     anidados (el primero es el bucle externo). El `x` de la expresión puede
#     usar variables de cualquiera de los for anteriores.
# [[... for fila in matriz] for i in range(n)]
#     Comprehension ANIDADA (una comprehension dentro de la expresión de
#     otra): produce lista de listas. No confundir con la anterior, que
#     produce una lista plana.
# zip(*matriz)
#     Transposición idiomática: el * desempaqueta las filas como argumentos
#     separados, zip toma el i-ésimo de cada una. Devuelve TUPLAS (de ahí el
#     list(t) para volver a listas). Trunca si las filas tienen distinto largo.
# len(matriz[0])
#     Ancho de la matriz asumiendo filas uniformes. Con matriz vacía lanza
#     IndexError; en código de producción validar antes.
# f"{p}{n}"
#     Concatenación por f-string: convierte cada valor con str() de forma
#     implícita, sin el TypeError de "A" + 1.
# ============================================================================

matriz = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

# --- Imperativo ---
aplanada_imp = []
for fila in matriz:
    for valor in fila:
        aplanada_imp.append(valor)

transpuesta_imp = []
for i in range(len(matriz[0])):
    nueva_fila = []
    for fila in matriz:
        nueva_fila.append(fila[i])
    transpuesta_imp.append(nueva_fila)

combinaciones_imp = []
for p in "AB":
    for n in (1, 2):
        combinaciones_imp.append(f"{p}{n}")

# --- Declarativo ---
aplanada_dec = [valor for fila in matriz for valor in fila]
transpuesta_dec = [[fila[i] for fila in matriz] for i in range(len(matriz[0]))]
transpuesta_zip = [list(t) for t in zip(*matriz)]      # la forma idiomática
combinaciones_dec = [f"{p}{n}" for p in "AB" for n in (1, 2)]

print("aplanar    :", aplanada_imp == aplanada_dec, aplanada_dec)
print("transponer :", transpuesta_imp == transpuesta_dec == transpuesta_zip, transpuesta_zip)
print("cartesiano :", combinaciones_imp == combinaciones_dec, combinaciones_dec)

aplanar    : True [1, 2, 3, 4, 5, 6, 7, 8, 9]
transponer : True [[1, 4, 7], [2, 5, 8], [3, 6, 9]]
cartesiano : True ['A1', 'A2', 'B1', 'B2']


## 13. `reduce()`: plegar una secuencia a un único valor

Aplica la función de a dos elementos arrastrando el acumulado. Es el caso
general de `sum`/`max`/`min`, para cuando la operación no tiene un built-in.

**Regla práctica:** si existe el built-in (`sum`, `max`, `"".join`), usarlo.
`reduce` se justifica cuando la operación es propia del dominio.

In [14]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# functools.reduce(funcion, iterable, [inicial])
#     Aplica funcion(acumulado, siguiente) de izquierda a derecha hasta dejar
#     UN valor. Sin `inicial`, el primer elemento es el acumulado de partida
#     y con iterable vacío lanza TypeError. CON `inicial` el vacío devuelve
#     `inicial`. Regla: pasar siempre el inicial (es el "elemento neutro":
#     0 para sumar, 1 para multiplicar, [] para concatenar).
# lambda acumulado, actual: ...
#     La función de reduce recibe SIEMPRE dos argumentos, en ese orden:
#     primero lo acumulado, después el elemento nuevo. Invertirlos es el
#     error típico cuando la operación no es conmutativa (resta, concatenar).
# operator.mul / operator.add
#     Alternativa sin lambda: reduce(operator.mul, valores, 1). Más legible y
#     más rápido. Para el producto también existe math.prod(iterable) (3.8+).
# *=
#     Asignación aumentada de producto: producto_imp *= n equivale a
#     producto_imp = producto_imp * n.
# except TypeError
#     reduce sobre vacío sin inicial lanza TypeError (no ValueError): el
#     mensaje exacto es "reduce() of empty iterable with no initial value".
# ============================================================================

valores = [3, 5, 2, 7]

# --- Imperativo ---
producto_imp = 1                    # el valor inicial, explícito
for n in valores:
    producto_imp *= n

carritos = [{"total": 1500}, {"total": 2300}, {"total": 800}]
suma_imp = 0
for c in carritos:
    suma_imp += c["total"]

# --- Declarativo ---
producto_dec = reduce(lambda acumulado, actual: acumulado * actual, valores)
suma_dec = reduce(lambda acc, c: acc + c["total"], carritos, 0)  # 3er arg = inicial

print("producto:", producto_imp, producto_dec)
print("suma    :", suma_imp, suma_dec)

# Sin valor inicial y con secuencia vacía, reduce lanza TypeError
try:
    reduce(lambda a, b: a * b, [])
except TypeError as error:
    print("sin inicial:", error)
print("con inicial:", reduce(lambda a, b: a * b, [], 1))

producto: 210 210
suma    : 4600 4600
sin inicial: reduce() of empty iterable with no initial value
con inicial: 1


## 14. `collections`: estructuras con el comportamiento ya declarado

`Counter` cuenta y `defaultdict` decide qué hacer con una clave inexistente.
Ambos eliminan el patrón `if clave in dic: ... else: ...`, que es donde
aparecen los `KeyError`.

In [15]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# collections.Counter(iterable_o_mapping)
#     Subclase de dict que cuenta hashables. Clave ausente devuelve 0 en vez
#     de KeyError. Soporta aritmética: c1 + c2 suma conteos, c1 - c2 resta
#     y descarta los <= 0, c1 & c2 mínimos, c1 | c2 máximos.
#     Counter.total() (3.10+) suma todos los conteos.
# Counter.most_common(n=None)
#     Lista de (elemento, conteo) ordenada de mayor a menor. Empates
#     conservan el orden de primera aparición. Sin n devuelve todos.
# collections.defaultdict(default_factory)
#     Dict que, al acceder a una clave AUSENTE, llama a default_factory()
#     sin argumentos, guarda el resultado y lo devuelve. list, int, set,
#     dict o cualquier callable sirven. Ojo: la simple LECTURA crea la clave
#     (d["x"] la inserta); usar `"x" in d` o d.get() para consultar sin crear.
# dict.get(clave, default=None)
#     Lectura sin KeyError: devuelve `default` si la clave no existe y NO la
#     inserta. El idioma d[k] = d.get(k, 0) + 1 es el Counter manual.
# sorted(d.items(), key=lambda par: par[1], reverse=True)[:2]
#     Top-N a mano: ordena pares por valor y rebana. Para N pequeño sobre
#     datos grandes es más eficiente heapq.nlargest(2, d.items(), key=...).
# dict(defaultdict)
#     Convierte a dict normal: útil para serializar o comparar (un defaultdict
#     == dict si tienen el mismo contenido, pero json.dumps y repr difieren).
# ============================================================================

palabras = ["pan", "leche", "pan", "queso", "pan", "leche"]
compras = [("Ana", "pan"), ("Beto", "leche"), ("Ana", "queso")]

# --- Imperativo ---
conteo_imp = {}
for palabra in palabras:
    if palabra in conteo_imp:        # la comprobación que sobra
        conteo_imp[palabra] += 1
    else:
        conteo_imp[palabra] = 1

top_imp = sorted(conteo_imp.items(), key=lambda par: par[1], reverse=True)[:2]

por_persona_imp = {}
for persona, item in compras:
    if persona not in por_persona_imp:
        por_persona_imp[persona] = []
    por_persona_imp[persona].append(item)

# --- Declarativo ---
conteo_dec = Counter(palabras)
top_dec = conteo_dec.most_common(2)

por_persona_dec = defaultdict(list)   # "una clave nueva nace como lista vacía"
for persona, item in compras:
    por_persona_dec[persona].append(item)

print("conteo :", conteo_imp == dict(conteo_dec), conteo_dec)
print("top 2  :", top_imp == top_dec, top_dec)
print("agrupar:", por_persona_imp == dict(por_persona_dec), dict(por_persona_dec))

conteo : True Counter({'pan': 3, 'leche': 2, 'queso': 1})
top 2  : True [('pan', 3), ('leche', 2)]
agrupar: True {'Ana': ['pan', 'queso'], 'Beto': ['leche']}


## 15. `itertools`: componer iteradores perezosos

- `chain` concatena sin copiar a una lista nueva.
- `islice` rebana cualquier iterable, incluso uno infinito.
- `accumulate` entrega los resultados **parciales**.
- `groupby` agrupa elementos **consecutivos** → hay que ordenar antes por la
  misma clave, o salen grupos repetidos. Es la trampa nº 1 de `itertools`.

In [16]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# itertools.chain(*iterables)
#     Concatena iterables en UN iterador perezoso sin copiar. Si los
#     iterables vienen en una lista, usar chain.from_iterable(lista) en vez
#     de chain(*lista) (evita desempaquetar miles de argumentos).
# itertools.islice(iterable, stop) / islice(iterable, start, stop, step)
#     Slicing perezoso sobre CUALQUIER iterable, incluso infinitos o sin
#     índices (generadores, archivos). No admite índices negativos.
#     Consume los elementos que salta: islice(it, 5, 10) descarta los 5
#     primeros del iterador original.
# itertools.accumulate(iterable, funcion=operator.add, *, initial=None)
#     Como reduce pero devuelve los resultados PARCIALES. Con funcion=max
#     da el máximo acumulado, con operator.mul el producto acumulado.
#     initial (3.8+) antepone un valor de partida.
# itertools.groupby(iterable, key=None)
#     Agrupa elementos CONSECUTIVOS con la misma key. NO ordena: si la
#     entrada no está ordenada por esa key, la misma clave aparece en varios
#     grupos (ver "sin ordenar" abajo). Cada `grupo` es un iterador que se
#     invalida al avanzar al siguiente: consumirlo (sum, list) antes de seguir.
# sorted(..., key=itemgetter("ciudad"))
#     El paso previo obligatorio de groupby. Misma key en ambos.
# enumerate(generador)
#     Funciona sobre cualquier iterable; aquí permite cortar con break un
#     generador de un millón sin materializarlo (imita islice a mano).
# dict.get(k, 0)      -> ver celda 14.
# ============================================================================

# --- Imperativo ---
cadena_imp = []
for iterable in ([1, 2], (3, 4), {5}):
    for item in iterable:
        cadena_imp.append(item)

primeros_imp = []
for i, valor in enumerate(n ** 2 for n in range(1_000_000)):
    if i >= 5:
        break
    primeros_imp.append(valor)

acumulado, parciales_imp = 0, []
for n in [10, 5, 20, 3]:
    acumulado += n
    parciales_imp.append(acumulado)

ventas = [
    {"ciudad": "Santiago", "monto": 100},
    {"ciudad": "Valparaíso", "monto": 40},
    {"ciudad": "Santiago", "monto": 60},
]
por_ciudad_imp = {}
for v in ventas:
    por_ciudad_imp[v["ciudad"]] = por_ciudad_imp.get(v["ciudad"], 0) + v["monto"]

# --- Declarativo ---
cadena_dec = list(chain([1, 2], (3, 4), {5}))
primeros_dec = list(islice((n ** 2 for n in range(1_000_000)), 5))
parciales_dec = list(accumulate([10, 5, 20, 3]))

ordenadas = sorted(ventas, key=itemgetter("ciudad"))   # ordenar ANTES de groupby
por_ciudad_dec = {
    ciudad: sum(v["monto"] for v in grupo)
    for ciudad, grupo in groupby(ordenadas, key=itemgetter("ciudad"))
}

print("chain     :", cadena_imp == cadena_dec, cadena_dec)
print("islice    :", primeros_imp == primeros_dec, primeros_dec)
print("accumulate:", parciales_imp == parciales_dec, parciales_dec)
print("groupby   :", por_ciudad_imp == por_ciudad_dec, por_ciudad_dec)

# Sin ordenar, groupby parte Santiago en dos grupos:
sin_ordenar = {c: len(list(g)) for c, g in groupby(ventas, key=itemgetter("ciudad"))}
print("groupby sin ordenar (mal):", sin_ordenar)

chain     : True [1, 2, 3, 4, 5]
islice    : True [0, 1, 4, 9, 16]
accumulate: True [10, 15, 35, 38]
groupby   : True {'Santiago': 160, 'Valparaíso': 40}
groupby sin ordenar (mal): {'Santiago': 1, 'Valparaíso': 1}


## 16. `dataclass`: declarar la forma del dato, no su plomería

El decorador genera `__init__`, `__repr__` y `__eq__` a partir de las
anotaciones. Comparar el largo de las dos clases dice todo.

- `frozen=True` → inmutable y hashable: nadie lo muta por accidente.
- `field(default_factory=list)` → obligatorio para mutables; una lista como
  valor por defecto se compartiría entre **todas** las instancias (bug clásico).
- `@property` → atributo calculado: se declara la fórmula, no se guarda el estado.

In [17]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# class Nombre: ... / def __init__(self, ...)
#     Clase clásica: __init__ inicializa la instancia (no la construye; eso
#     es __new__). `self` es la instancia y se pasa implícitamente.
# __repr__(self) -> str
#     Representación para desarrolladores (lo que muestra la consola y las
#     listas). Convención: parecerse a la expresión que reconstruye el objeto.
#     !r dentro de la f-string llama a repr() del valor (pone las comillas).
# __eq__(self, otro)
#     Define ==. Devolver NotImplemented (no False) cuando el otro tipo es
#     desconocido: así Python prueba con el __eq__ del otro operando.
#     Definir __eq__ sin __hash__ hace la clase NO hashable (no sirve como
#     clave de dict ni en sets): por eso se define __hash__ a mano.
# __hash__(self) -> int
#     Debe ser consistente con __eq__: objetos iguales, mismo hash. Se
#     calcula sobre una tupla de los campos.
# @dataclass(frozen=False, order=False, eq=True, slots=False, kw_only=False)
#     Genera __init__, __repr__, __eq__ (y __hash__ si frozen=True) a partir
#     de las ANOTACIONES de tipo (sin anotación, no es campo).
#     order=True añade <, <=, >, >= comparando los campos en orden de
#     declaración. slots=True (3.10+) ahorra memoria y bloquea atributos no
#     declarados. kw_only=True obliga a pasar los campos por nombre.
#     frozen=True: asignar un campo lanza dataclasses.FrozenInstanceError.
# dataclasses.field(default_factory=..., default=..., repr=True, compare=True)
#     Configura un campo. default_factory es OBLIGATORIO para defaults
#     mutables: `items: list = []` es ValueError en dataclass justamente para
#     evitar la lista compartida entre instancias.
# tuple[str, ...]
#     Anotación: tupla de largo variable de str. Se usa tupla y no lista para
#     que el dataclass frozen sea realmente hashable (una lista dentro rompe
#     hash()).
# @property
#     Convierte un método en atributo de solo lectura calculado al vuelo:
#     carrito.total, sin paréntesis. No se cachea: se recalcula en cada
#     acceso (para cachear: functools.cached_property).
# isinstance(objeto, clase_o_tupla)
#     True si es instancia de la clase o de una subclase. Preferirlo a
#     type(x) == Clase, que ignora la herencia.
# operator.attrgetter("precio")    -> ver celda 7.
# except Exception as error / type(error).__name__
#     Se captura genérico solo para MOSTRAR el nombre de la excepción real.
#     En código de producción se captura la clase concreta (FrozenInstanceError).
# ============================================================================

# --- Imperativo: todo escrito a mano ---
class ProductoImp:
    def __init__(self, nombre, precio, etiquetas=()):
        self.nombre = nombre
        self.precio = precio
        self.etiquetas = etiquetas

    def __repr__(self):
        return (f"ProductoImp(nombre={self.nombre!r}, precio={self.precio!r}, "
                f"etiquetas={self.etiquetas!r})")

    def __eq__(self, otro):
        if not isinstance(otro, ProductoImp):
            return NotImplemented
        return ((self.nombre, self.precio, self.etiquetas)
                == (otro.nombre, otro.precio, otro.etiquetas))

    def __hash__(self):
        return hash((self.nombre, self.precio, self.etiquetas))


# --- Declarativo: las mismas capacidades en 4 líneas ---
@dataclass(frozen=True, order=True)
class Producto:
    nombre: str
    precio: int
    etiquetas: tuple[str, ...] = ()


@dataclass
class Carrito:
    items: list[Producto] = field(default_factory=list)

    @property
    def total(self) -> int:
        """Atributo calculado: la fórmula se declara, el valor no se almacena."""
        return sum(p.precio for p in self.items)


catalogo = [
    Producto("pan", 1200, ("panaderia",)),
    Producto("queso", 4500, ("lacteos", "premium")),
    Producto("leche", 990, ("lacteos",)),
]

print("repr  :", catalogo[0])
print("eq    :", Producto("pan", 1200, ("panaderia",)) == catalogo[0])
print("total :", Carrito(items=catalogo).total)
print("orden :", [p.nombre for p in sorted(catalogo, key=attrgetter("precio"))])

# frozen=True impide la mutación accidental
try:
    catalogo[0].precio = 1
except Exception as error:
    print("inmutable:", type(error).__name__, error)

repr  : Producto(nombre='pan', precio=1200, etiquetas=('panaderia',))
eq    : True
total : 6690
orden : ['leche', 'pan', 'queso']
inmutable: FrozenInstanceError cannot assign to field 'precio'


## 17. Pattern matching estructural (`match`/`case`, 3.10+)

No es un `switch`: describe la **forma** que deben tener los datos y
desempaqueta en variables las partes que interesan. Reemplaza cadenas de
`if/elif` llenas de `isinstance()`, `len()`, `in` e índices.

In [18]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# class Estado(Enum)
#     Enumeración: miembros con nombre y valor únicos. Estado.OK es un
#     singleton: se compara con `is` (identidad) o ==. Estado("ok") busca
#     por valor, Estado["OK"] por nombre. Iterar la clase recorre los miembros.
# match sujeto: / case patron:
#     Pattern matching estructural (PEP 634, 3.10+). Prueba los case EN
#     ORDEN y ejecuta el primero que encaja; no hay "fall-through" ni break.
#     Los nombres del patrón se ASIGNAN (captura), no se comparan: `case x:`
#     captura cualquier cosa. Para comparar contra una variable usar un nombre
#     con punto (Estado.OK) o un literal.
# case {"tipo": "click", "x": x, "y": y}
#     Patrón de MAPPING: encaja si el sujeto es un dict (o Mapping) con AL
#     MENOS esas claves; claves extra se ignoran. Los valores pueden ser
#     literales (comparan) o nombres (capturan). **resto captura las demás.
# case {"valor": str(v)}
#     Patrón de CLASE: str(v) exige isinstance(valor, str) y captura en v.
#     int(), float(), list() funcionan igual. Con dataclasses se pueden
#     desempaquetar campos: case Producto(nombre=n, precio=p).
# case ... if condicion
#     GUARDA: el case encaja solo si además la condición es verdadera. Las
#     capturas ya están disponibles dentro de la guarda (v.isdigit()).
# case [Estado.OK, *resto]
#     Patrón de SECUENCIA: encaja list/tuple (NO str ni bytes) cuyo primer
#     elemento == Estado.OK; *resto captura el resto como lista.
#     [] encaja la secuencia vacía, [_] exactamente un elemento.
# case (int() | float()) as numero
#     | combina alternativas (OR de patrones). `as` captura el valor completo
#     que encajó. Cuidado: bool es subclase de int, así que True encaja en
#     int(); el imperativo lo excluye a mano con `not isinstance(e, bool)`.
# case _
#     Comodín: encaja siempre y no captura. Sin él, si nada encaja, match no
#     hace nada (no lanza error): ponerlo para no tragar casos por error.
# str.isdigit()
#     True si TODOS los caracteres son dígitos y hay al menos uno. Acepta
#     dígitos Unicode ("٣", "²"); para ASCII estricto: s.isascii() and
#     s.isdigit(). "-7".isdigit() es False (el guion no es dígito).
# dict.get(k) / "x" in evento / isinstance(e, (int, float))
#     La plomería que match reemplaza: isinstance acepta una tupla de tipos.
# f"{imp:<22}"
#     Formato de alineación: < alinea a la izquierda en 22 columnas, > a la
#     derecha, ^ centra. Útil para tablas en consola.
# ============================================================================

class Estado(Enum):
    OK = "ok"
    ERROR = "error"


# --- Imperativo: comprobar tipo, claves y largo a mano ---
def describir_imp(evento) -> str:
    if isinstance(evento, dict) and evento.get("tipo") == "click" \
            and "x" in evento and "y" in evento:
        return f"click en ({evento['x']}, {evento['y']})"
    if isinstance(evento, dict) and evento.get("tipo") == "tecla":
        valor = evento.get("valor")
        if isinstance(valor, str) and valor.isdigit():
            return f"dígito {valor}"
        return f"tecla {valor}"
    if isinstance(evento, list) and len(evento) >= 1 and evento[0] is Estado.OK:
        return f"éxito con {len(evento) - 1} extras"
    if isinstance(evento, (int, float)) and not isinstance(evento, bool):
        return f"número {evento}"
    return "evento desconocido"


# --- Declarativo: se describe la forma, Python hace el resto ---
def describir_dec(evento) -> str:
    match evento:
        case {"tipo": "click", "x": x, "y": y}:                  # dict con esas claves
            return f"click en ({x}, {y})"
        case {"tipo": "tecla", "valor": str(v)} if v.isdigit():  # patrón + guarda
            return f"dígito {v}"
        case {"tipo": "tecla", "valor": v}:
            return f"tecla {v}"
        case [Estado.OK, *resto]:                                # lista que parte en OK
            return f"éxito con {len(resto)} extras"
        case (int() | float()) as numero:                        # alternativas con |
            return f"número {numero}"
        case _:                                                  # comodín: el default
            return "evento desconocido"


eventos = [
    {"tipo": "click", "x": 10, "y": 20},
    {"tipo": "tecla", "valor": "7"},
    {"tipo": "tecla", "valor": "Esc"},
    [Estado.OK, "a", "b"],
    3.14,
    None,
]
for evento in eventos:
    imp, dec = describir_imp(evento), describir_dec(evento)
    print(f"{imp:<22} | {dec:<22} | iguales: {imp == dec}")

click en (10, 20)      | click en (10, 20)      | iguales: True
dígito 7               | dígito 7               | iguales: True
tecla Esc              | tecla Esc              | iguales: True
éxito con 2 extras     | éxito con 2 extras     | iguales: True
número 3.14            | número 3.14            | iguales: True
evento desconocido     | evento desconocido     | iguales: True


## 18. Decoradores: declarar comportamiento *alrededor* de la función

`@lru_cache` memoiza. La función sigue escrita como la definición matemática
pura; el caché es una **declaración puesta encima**, no código mezclado en el
cuerpo. Compará los tiempos: el mismo cálculo, órdenes de magnitud de
diferencia.

In [19]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# @decorador
#     Azúcar sintáctico: `@d` sobre `def f` equivale a f = d(f). El decorador
#     recibe la función y devuelve otra (o la misma). Se aplica una vez, al
#     definir. Los que devuelven un wrapper deben usar functools.wraps para
#     conservar __name__ y __doc__.
# functools.lru_cache(maxsize=128, typed=False)
#     Memoiza por ARGUMENTOS (deben ser hashables: listas no sirven). Con
#     maxsize=None guarda todo (= functools.cache, 3.9+); con un número
#     descarta el menos usado recientemente (LRU). Las llamadas con el mismo
#     valor pero distinto tipo (1 vs 1.0) comparten entrada salvo typed=True.
#     En métodos de instancia el caché retiene `self`: puede fugar memoria.
# f.cache_info()
#     Namedtuple (hits, misses, maxsize, currsize). hits = respuestas servidas
#     sin ejecutar la función. f.cache_clear() vacía el caché.
# _memo = {} / n in _memo
#     Memoización manual: el prefijo _ marca "privado por convención". `in`
#     sobre dict es O(1). Es exactamente lo que lru_cache hace, pero mezclado
#     con la lógica de negocio.
# n if n < 2 else ...
#     Ternario como cuerpo completo de la función: la definición recursiva de
#     Fibonacci en una línea. Sin caché es O(2^n): fib(30) hace ~2.7 millones
#     de llamadas; con caché, una sola por cada n distinto (31 en total).
# time.perf_counter()
#     Reloj monotónico de alta resolución en segundos (float). Solo tiene
#     sentido la DIFERENCIA entre dos lecturas, no el valor absoluto. Usarlo
#     en vez de time.time() para medir duraciones (no salta con cambios de
#     hora del sistema). Para benchmarks serios: módulo timeit.
# ============================================================================

# --- Imperativo: la memoización a mano, mezclada con la lógica ---
_memo = {}


def fib_imp(n: int) -> int:
    if n in _memo:                    # plomería del caché...
        return _memo[n]
    resultado = n if n < 2 else fib_imp(n - 1) + fib_imp(n - 2)
    _memo[n] = resultado              # ...repartida por toda la función
    return resultado


# --- Sin caché: la misma definición, exponencial ---
def fib_lento(n: int) -> int:
    return n if n < 2 else fib_lento(n - 1) + fib_lento(n - 2)


# --- Declarativo: la lógica queda limpia, el caché es una línea aparte ---
@lru_cache(maxsize=None)
def fib_dec(n: int) -> int:
    return n if n < 2 else fib_dec(n - 1) + fib_dec(n - 2)


print("primeros 10:", [fib_dec(n) for n in range(10)])
print("fib(90)    :", fib_imp(90), fib_dec(90), fib_imp(90) == fib_dec(90))

inicio = time.perf_counter()
fib_lento(30)
sin_cache = time.perf_counter() - inicio

inicio = time.perf_counter()
fib_dec(30)
con_cache = time.perf_counter() - inicio

print(f"fib(30) sin caché: {sin_cache:.4f}s")
print(f"fib(30) con caché: {con_cache:.6f}s")
print("estado del caché  :", fib_dec.cache_info())

primeros 10: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
fib(90)    : 2880067194370816120 2880067194370816120 True
fib(30) sin caché: 0.1076s
fib(30) con caché: 0.000032s
estado del caché  : CacheInfo(hits=100, misses=91, maxsize=None, currsize=91)


## 19. Context managers: declarar el ciclo de vida de un recurso

`with` garantiza la limpieza **aunque salte una excepción**, sin repetir
`try/finally` en cada uso. `@contextmanager` construye uno con un generador:
lo anterior al `yield` es la entrada, lo del `finally` es la salida.

In [20]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# try: ... finally: ...
#     El bloque finally se ejecuta SIEMPRE: con éxito, con excepción, con
#     return o break dentro del try. Es la garantía que `with` empaqueta.
# with gestor as valor:
#     Llama a gestor.__enter__() al entrar (su retorno va a `valor`) y a
#     gestor.__exit__(tipo, exc, tb) al salir, pase lo que pase. Si __exit__
#     devuelve True la excepción se SUPRIME; si devuelve None/False se
#     propaga. Varios gestores: with a() as x, b() as y: (3.10+ admite
#     paréntesis para partir en líneas).
# @contextlib.contextmanager
#     Convierte una función generadora en gestor de contexto. Lo anterior al
#     `yield` corre en __enter__; lo que se hace yield es el `as valor`; lo
#     posterior corre en __exit__. Si el bloque with lanza, la excepción se
#     RE-LANZA en el yield: por eso hace falta try/finally (o try/except)
#     alrededor del yield para que la limpieza corra. Debe hacer yield
#     EXACTAMENTE una vez.
# yield
#     Convierte la función en generadora. Sin valor, produce None. Suspende
#     la ejecución en ese punto y la reanuda cuando el consumidor lo pide.
# raise ValueError("mensaje")
#     Lanza una excepción. Dentro de un with, dispara __exit__ (aquí el
#     finally del cronómetro) y luego sigue subiendo hasta el except externo.
# time.perf_counter()     -> ver celda 18.
# ============================================================================

# --- Imperativo: try/finally repetido en cada punto de uso ---
inicio = time.perf_counter()
try:
    total_imp = sum(n for n in range(1_000_000))
finally:
    print(f"imperativo : tardó {time.perf_counter() - inicio:.4f}s")


# --- Declarativo: el protocolo se declara una vez y se reutiliza ---
@contextmanager
def cronometro(etiqueta: str):
    inicio = time.perf_counter()
    try:
        yield                     # acá se ejecuta el bloque del `with`
    finally:
        print(f"{etiqueta}: tardó {time.perf_counter() - inicio:.4f}s")


with cronometro("declarativo"):
    total_dec = sum(n for n in range(1_000_000))

print("mismo resultado:", total_imp == total_dec, total_dec)

# La limpieza corre igual cuando hay excepción
try:
    with cronometro("con excepción"):
        raise ValueError("algo falló")
except ValueError as error:
    print("excepción propagada:", error)

imperativo : tardó 0.0249s
declarativo: tardó 0.0266s
mismo resultado: True 499999500000
con excepción: tardó 0.0000s
excepción propagada: algo falló


## 20. Pipeline declarativo: componer generadores como etapas

Cada etapa describe **una** transformación y devuelve un generador. Al
encadenarlas se obtiene un flujo perezoso: nada se calcula hasta que el
consumidor final pide datos, y la memoria no depende del tamaño de la entrada.

Es el patrón detrás de un ETL o del procesamiento de un archivo enorme. La
versión imperativa mete parseo, validación, enriquecimiento y agregación en un
solo bucle: imposible de probar por partes y de reordenar.

In [21]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# def funcion(...): ... yield ...
#     Función GENERADORA: al llamarla no ejecuta nada, devuelve un generador.
#     Cada yield entrega un valor y pausa; `return` (o llegar al final)
#     termina el flujo. Ideal para etapas de pipeline: procesa de a uno y no
#     retiene la colección completa.
# continue
#     Salta al siguiente elemento del bucle. En parsear() implementa el
#     "descartar lo inválido" sin anidar ifs.
# str.split(",")
#     Con separador explícito: NO colapsa vacíos ("a,,b" -> 3 partes) y un
#     string sin comas devuelve [string] (1 parte). De ahí el len(partes) != 3.
#     Para CSV real usar el módulo csv (maneja comillas y comas internas).
# str.lstrip(caracteres)
#     Quita del INICIO todos los caracteres que estén en el argumento (es un
#     conjunto, no un prefijo): "--7".lstrip("-") -> "7". Para quitar un
#     prefijo exacto: str.removeprefix("-") (3.9+). rstrip/strip: final/ambos.
# str.isdigit()           -> ver celda 17. Aquí, tras quitar el signo, valida
#     que el resto sea un entero. int() lanzaría ValueError con "abc".
# int(cadena)
#     Parsea un entero; acepta signo y espacios alrededor ("  -20 " -> -20),
#     rechaza decimales ("1.5" -> ValueError). int("1_000") es válido.
# fecha[:7]
#     Slicing de str: los 7 primeros caracteres ("2026-01"). Nunca lanza
#     IndexError aunque el string sea más corto.
# {**fila, "mes": ...}
#     Copia superficial + campo nuevo, sin mutar la fila original. Las etapas
#     NO deben mutar lo que reciben: otras etapas pueden tener referencia.
# collections.defaultdict(int)
#     int() sin argumentos devuelve 0: la clave nueva nace en 0 y `+=` suma
#     directo. Ver celda 14.
# dict(defaultdict)       -> ver celda 14.
# resumir(enriquecer(validos(parsear(registros))))
#     Composición por anidamiento: se lee de ADENTRO hacia afuera. Nada se
#     ejecuta hasta que resumir() itera; recién ahí cada fila atraviesa las
#     cuatro etapas de a una.
# def componer(*etapas)
#     *etapas recibe cualquier cantidad de funciones como tupla. Devuelve una
#     lambda (clausura) que recuerda esas etapas. reduce las aplica en orden:
#     etapa(datos) con el resultado anterior como entrada. componer(f, g)(x)
#     es g(f(x)): orden de lectura natural, al revés de la notación matemática.
# len(list(generador))
#     Materializa un generador solo para contarlo. Alternativa sin lista:
#     sum(1 for _ in generador). Ambas CONSUMEN el generador.
# ============================================================================

registros = [
    "2026-01-05,Santiago,1500",
    "2026-01-06,Valparaíso,-20",  # monto negativo: se descarta al validar
    "linea corrompida",           # formato inválido: se descarta al parsear
    "2026-01-07,Santiago,3200",
    "2026-02-01,Concepción,800",
]

# --- Imperativo: las 4 responsabilidades en un solo bucle ---
resultado_imp = {}
for linea in registros:
    partes = linea.split(",")
    if len(partes) != 3:
        continue
    fecha, ciudad, monto = partes
    if not monto.lstrip("-").isdigit():
        continue
    monto = int(monto)
    if monto <= 0:
        continue
    mes = fecha[:7]                                    # dato derivado
    resultado_imp[ciudad] = resultado_imp.get(ciudad, 0) + monto


# --- Declarativo: una función por etapa, cada una testeable por separado ---
def parsear(lineas):
    """Etapa 1: texto -> dict. Descarta lo que no calce con el formato."""
    for linea in lineas:
        partes = linea.split(",")
        if len(partes) != 3:
            continue
        fecha, ciudad, monto = partes
        if not monto.lstrip("-").isdigit():
            continue
        yield {"fecha": fecha, "ciudad": ciudad, "monto": int(monto)}


def validos(filas):
    """Etapa 2: dejar solo montos positivos."""
    return (fila for fila in filas if fila["monto"] > 0)


def enriquecer(filas):
    """Etapa 3: agregar columnas derivadas."""
    return ({**fila, "mes": fila["fecha"][:7]} for fila in filas)


def resumir(filas) -> dict[str, int]:
    """Etapa 4: colapsar el flujo en un agregado por ciudad."""
    acumulado = defaultdict(int)
    for fila in filas:
        acumulado[fila["ciudad"]] += fila["monto"]
    return dict(acumulado)


resultado_dec = resumir(enriquecer(validos(parsear(registros))))


def componer(*etapas):
    """Compone de izquierda a derecha: componer(f, g)(x) == g(f(x))."""
    return lambda entrada: reduce(lambda datos, etapa: etapa(datos), etapas, entrada)


procesar = componer(parsear, validos, enriquecer, resumir)

print("imperativo :", resultado_imp)
print("declarativo:", resultado_dec)
print("compuesto  :", procesar(registros))
print("los tres iguales:", resultado_imp == resultado_dec == procesar(registros))

# Ventaja concreta: cada etapa se prueba sola, sin tocar las demás
print("solo etapa 1:", len(list(parsear(registros))), "filas parseadas")

imperativo : {'Santiago': 4700, 'Concepción': 800}
declarativo: {'Santiago': 4700, 'Concepción': 800}
compuesto  : {'Santiago': 4700, 'Concepción': 800}
los tres iguales: True
solo etapa 1: 4 filas parseadas


## Resumen: qué reemplaza a qué

| Patrón imperativo | Equivalente declarativo |
|---|---|
| lista vacía + `for` + `append` | list comprehension |
| `if` dentro del bucle para saltar | `if` al final de la comprehension |
| dict vacío + asignación en bucle | dict comprehension |
| `if x not in resultado` | set comprehension |
| clase con `__iter__`/`__next__` | generator expression |
| acumulador / bandera booleana | `sum`, `min`, `max`, `any`, `all` |
| escribir un algoritmo de orden | `sorted(key=...)` |
| bucle que transforma | `map` / comprehension |
| bucle que descarta | `filter` / comprehension con `if` |
| `for i in range(len(...))` | `zip`, `enumerate` |
| acceso por índice, `update()` | unpacking `*`, `**` |
| `if/else` con dos `append` | ternario dentro de la comprehension |
| bucles anidados | comprehension con varios `for`, `zip(*m)` |
| acumulador con operación propia | `reduce` |
| `if clave in dic: ... else: ...` | `Counter`, `defaultdict` |
| bucles con `break` y contadores | `itertools` |
| `__init__`/`__repr__`/`__eq__` a mano | `@dataclass` |
| cadena de `if isinstance(...)` | `match`/`case` |
| caché mezclado con la lógica | `@lru_cache` |
| `try/finally` repetido | `with` + `@contextmanager` |
| un bucle que hace todo | pipeline de generadores |

**Cuándo NO ser declarativo:** si la comprehension necesita más de dos `for`
o una condición que no cabe en una línea, un bucle con nombre se lee mejor.
Lo declarativo deja de pagar cuando la expresión hay que releerla dos veces.

In [22]:
# ============================================================================
# MÉTODOS Y CONSTRUCCIONES USADOS EN ESTA CELDA
# ============================================================================
# dict literal con claves de texto
#     Tabla de comprobaciones: cada valor es un bool ya evaluado. Se usa un
#     dict (no una lista) para que el nombre de cada chequeo viaje junto al
#     resultado.
# and
#     Cortocircuito: si el primer operando es falsy devuelve ESE operando sin
#     evaluar el segundo (no devuelve necesariamente True/False). Aquí se
#     combinan varias igualdades en un solo bool.
# all(comprobaciones.values())
#     dict.values() es la vista perezosa de los valores; all() devuelve True
#     solo si todos son verdaderos. Con dict vacío devolvería True (verdad
#     vacía): no confundir "todo pasó" con "no se comprobó nada".
# 'OK ' if ok else 'FALLA'
#     Ternario dentro de una f-string: las expresiones interpoladas pueden
#     ser cualquier expresión, incluidas condicionales (entre llaves, sin
#     comillas del mismo tipo que la f-string hasta 3.12; desde 3.12, PEP 701,
#     se pueden reutilizar).
# "\n"
#     Secuencia de escape: salto de línea. Dentro de un string normal se
#     interpreta; en un raw string (r"...") quedaría literal.
# ============================================================================

# Chequeo final: recorrer los resultados clave del notebook
comprobaciones = {
    "1  comprehension": cuadrados_imp == cuadrados_dec,
    "2  filtro": pares_imp == pares_dec,
    "3  dict comp": con_descuento_imp == con_descuento_dec,
    "4  set comp": sorted(iniciales_imp) == sorted(iniciales_dec),
    "6  agregadoras": promedio_imp == promedio_dec and hay_reprobada_imp == hay_reprobada_dec,
    "7  sorted": [a["nombre"] for a in por_nota_imp] == [a["nombre"] for a in por_nota_dec],
    "8  map/filter": fahrenheit_imp == fahrenheit_dec and calurosas_imp == calurosas_dec,
    "9  zip": tabla_imp == tabla_dec,
    "10 unpacking": config_imp == config_dec,
    "11 ternario": estado_imp == estado_dec and aprobados_imp == aprobados_dec,
    "12 anidadas": aplanada_imp == aplanada_dec and transpuesta_imp == transpuesta_zip,
    "13 reduce": producto_imp == producto_dec and suma_imp == suma_dec,
    "14 collections": conteo_imp == dict(conteo_dec),
    "15 itertools": por_ciudad_imp == por_ciudad_dec,
    "17 match/case": all(describir_imp(e) == describir_dec(e) for e in eventos),
    "18 lru_cache": fib_imp(90) == fib_dec(90),
    "19 with": total_imp == total_dec,
    "20 pipeline": resultado_imp == resultado_dec,
}
for etiqueta, ok in comprobaciones.items():
    print(f"{'OK ' if ok else 'FALLA'} {etiqueta}")
print("\nTodo equivalente:", all(comprobaciones.values()))

OK  1  comprehension
OK  2  filtro
OK  3  dict comp
OK  4  set comp
OK  6  agregadoras
OK  7  sorted
OK  8  map/filter
OK  9  zip
OK  10 unpacking
OK  11 ternario
OK  12 anidadas
OK  13 reduce
OK  14 collections
OK  15 itertools
OK  17 match/case
OK  18 lru_cache
OK  19 with
OK  20 pipeline

Todo equivalente: True
